In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

**Tiny Input**

In [3]:
batch_size = 1
seq_len = 4
d_model = 8

x = torch.rand(batch_size, seq_len, d_model)
print("Input shape :", x.shape)
print(x)

Input shape : torch.Size([1, 4, 8])
tensor([[[0.8860, 0.5832, 0.3376, 0.8090, 0.5779, 0.9040, 0.5547, 0.3423],
         [0.6343, 0.3644, 0.7104, 0.9464, 0.7890, 0.2814, 0.7886, 0.5895],
         [0.7539, 0.1952, 0.0050, 0.3068, 0.1165, 0.9103, 0.6440, 0.7071],
         [0.6581, 0.4913, 0.8913, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278]]])


**Single scaled dot-product attention function**

In [4]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    print("\n--- Inside scaled_dot_product_attention ---")
    print("Q shape:", Q.shape)
    print("K shape:", K.shape)
    print("V shape:", V.shape)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    print("\nAttention scores shape:", scores.shape)
    print(scores)

    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
        print("\nMasked scores:")
        print(scores)

    attention_weights = torch.softmax(scores, dim=-1)
    print("\nAttention weights shape:", attention_weights.shape)
    print(attention_weights)
    
    output = torch.matmul(attention_weights, V)
    print("\nAttention output shape:", output.shape)
    print(output)
    
    return output, attention_weights

In [5]:
Q = x
K = x
V = x

output, attention_weights = scaled_dot_product_attention(Q, K, V)


--- Inside scaled_dot_product_attention ---
Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])

Attention scores shape: torch.Size([1, 4, 4])
tensor([[[1.2267, 1.1065, 0.8914, 0.7826],
         [1.1065, 1.2752, 0.7482, 0.8980],
         [0.8914, 0.7482, 0.8689, 0.5305],
         [0.7826, 0.8980, 0.5305, 0.8248]]])

Attention weights shape: torch.Size([1, 4, 4])
tensor([[[0.3083, 0.2734, 0.2205, 0.1978],
         [0.2707, 0.3204, 0.1892, 0.2197],
         [0.2824, 0.2447, 0.2761, 0.1968],
         [0.2536, 0.2847, 0.1971, 0.2646]]])

Attention output shape: torch.Size([1, 4, 8])
tensor([[[0.7430, 0.4197, 0.4757, 0.6045, 0.5247, 0.5878, 0.6580, 0.4875],
         [0.7303, 0.4195, 0.5158, 0.6121, 0.5481, 0.5419, 0.6684, 0.4873],
         [0.7431, 0.4045, 0.4460, 0.5732, 0.4930, 0.6067, 0.6562, 0.5007],
         [0.7280, 0.4201, 0.5247, 0.5734, 0.5348, 0.5308, 0.6652, 0.4807]]])


**learnable projection**

In [6]:
W_q = nn.Linear(d_model, d_model)
W_k = nn.Linear(d_model, d_model)
W_v = nn.Linear(d_model, d_model)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

print("Projected Q shape:", Q.shape)
print("Projected K shape:", K.shape)
print("Projected V shape:", V.shape)

Projected Q shape: torch.Size([1, 4, 8])
Projected K shape: torch.Size([1, 4, 8])
Projected V shape: torch.Size([1, 4, 8])


In [7]:
output, attention_weights = scaled_dot_product_attention(Q, K, V)


--- Inside scaled_dot_product_attention ---
Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])

Attention scores shape: torch.Size([1, 4, 4])
tensor([[[ 0.0303,  0.1118,  0.1073,  0.0392],
         [-0.0447,  0.0550,  0.0634,  0.0239],
         [-0.1174,  0.0379, -0.1388, -0.0249],
         [-0.1392, -0.0172, -0.0512, -0.0112]]], grad_fn=<DivBackward0>)

Attention weights shape: torch.Size([1, 4, 4])
tensor([[[0.2396, 0.2599, 0.2588, 0.2417],
         [0.2331, 0.2575, 0.2597, 0.2497],
         [0.2356, 0.2752, 0.2307, 0.2585],
         [0.2294, 0.2592, 0.2505, 0.2608]]], grad_fn=<SoftmaxBackward0>)

Attention output shape: torch.Size([1, 4, 8])
tensor([[[ 2.3898e-01,  1.4964e-01, -6.3270e-03, -2.1874e-01, -6.6917e-01,
          -4.7470e-01,  4.7456e-01, -4.1689e-01],
         [ 2.4000e-01,  1.5005e-01, -4.2492e-03, -2.1787e-01, -6.6718e-01,
          -4.7255e-01,  4.7504e-01, -4.1581e-01],
         [ 2.4392e-01,  1.4254e-01,  5.0429e-03, -2.05

**Head split function**

In [8]:
def split_heads(x, num_heads):
    batch_size, seq_len, d_model = x.shape
    head_dim = d_model // num_heads
    
    x = x.view(batch_size, seq_len, num_heads, head_dim)
    x = x.transpose(1, 2)
    
    return x

In [9]:
num_heads = 2
head_dim = d_model // num_heads

Q_heads = split_heads(Q, num_heads)
K_heads = split_heads(K, num_heads)
V_heads = split_heads(V, num_heads)

print("Q_heads shape:", Q_heads.shape)
print("K_heads shape:", K_heads.shape)
print("V_heads shape:", V_heads.shape)

Q_heads shape: torch.Size([1, 2, 4, 4])
K_heads shape: torch.Size([1, 2, 4, 4])
V_heads shape: torch.Size([1, 2, 4, 4])


**Multi-head attention core**

In [10]:
output_heads, attention_weights = scaled_dot_product_attention(Q_heads, K_heads, V_heads)


--- Inside scaled_dot_product_attention ---
Q shape: torch.Size([1, 2, 4, 4])
K shape: torch.Size([1, 2, 4, 4])
V shape: torch.Size([1, 2, 4, 4])

Attention scores shape: torch.Size([1, 2, 4, 4])
tensor([[[[-0.0750, -0.0598, -0.0414, -0.0310],
          [-0.2054, -0.1654, -0.1281, -0.0983],
          [-0.1741, -0.0311, -0.2616, -0.0769],
          [-0.2797, -0.1798, -0.2105, -0.0938]],

         [[ 0.1179,  0.2179,  0.1931,  0.0864],
          [ 0.1422,  0.2432,  0.2177,  0.1321],
          [ 0.0080,  0.0847,  0.0654,  0.0417],
          [ 0.0828,  0.1555,  0.1381,  0.0780]]]], grad_fn=<DivBackward0>)

Attention weights shape: torch.Size([1, 2, 4, 4])
tensor([[[[0.2442, 0.2480, 0.2526, 0.2552],
          [0.2362, 0.2458, 0.2552, 0.2629],
          [0.2397, 0.2765, 0.2196, 0.2642],
          [0.2283, 0.2522, 0.2446, 0.2749]],

         [[0.2408, 0.2662, 0.2596, 0.2334],
          [0.2395, 0.2650, 0.2583, 0.2371],
          [0.2396, 0.2587, 0.2538, 0.2478],
          [0.2423, 0.2605, 0.

**Heads merge function**

In [15]:
def combine_heads(x):
    batch_size, num_heads, seq_len, head_dim = x.shape
    
    x = x.transpose(1, 2).contiguous()
    x = x.view(batch_size, seq_len, num_heads * head_dim)
    
    return x

In [16]:
combined_output = combine_heads(output_heads)

print("Combined output shape:", combined_output.shape)
print(combined_output)

Combined output shape: torch.Size([1, 4, 8])
tensor([[[ 0.2382,  0.1512, -0.0052, -0.2141, -0.6706, -0.4774,  0.4749,
          -0.4184],
         [ 0.2393,  0.1517, -0.0031, -0.2139, -0.6701, -0.4764,  0.4755,
          -0.4176],
         [ 0.2445,  0.1408,  0.0077, -0.2002, -0.6689, -0.4731,  0.4764,
          -0.4149],
         [ 0.2425,  0.1487,  0.0034, -0.2080, -0.6699, -0.4749,  0.4753,
          -0.4164]]], grad_fn=<ViewBackward0>)


**Final output projection**

In [17]:
W_o = nn.Linear(d_model, d_model)

final_output = W_o(combined_output)
print("Final output shape:", final_output.shape)
print(final_output)

Final output shape: torch.Size([1, 4, 8])
tensor([[[-0.4300,  0.4260,  0.3495, -0.0569, -0.6926,  0.2926,  0.5147,
           0.3254],
         [-0.4304,  0.4256,  0.3486, -0.0570, -0.6917,  0.2920,  0.5142,
           0.3246],
         [-0.4313,  0.4234,  0.3441, -0.0613, -0.6883,  0.2945,  0.5154,
           0.3189],
         [-0.4311,  0.4247,  0.3463, -0.0591, -0.6897,  0.2934,  0.5147,
           0.3222]]], grad_fn=<ViewBackward0>)


### Full MultiHeadAttention Class

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_len, d_model = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, num_heads * head_dim)
        return x

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

    def forward(self, x, mask=None):
        print("\n===== MultiHeadAttention Forward =====")
        print("Input x shape:", x.shape)

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        print("\nAfter linear projection:")
        print("Q shape:", Q.shape)
        print("K shape:", K.shape)
        print("V shape:", V.shape)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        print("\nAfter split_heads:")
        print("Q shape:", Q.shape)
        print("K shape:", K.shape)
        print("V shape:", V.shape)

        output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        print("\nAfter attention:")
        print("Output per head shape:", output.shape)
        print("Attention weights shape:", attention_weights.shape)

        output = self.combine_heads(output)

        print("\nAfter combine_heads:")
        print("Combined output shape:", output.shape)

        output = self.W_o(output)

        print("\nAfter final linear projection:")
        print("Final output shape:", output.shape)

        return output, attention_weights

In [19]:
mha = MultiHeadAttention(d_model=8, num_heads=2)

final_output, attention_weights = mha(x)


===== MultiHeadAttention Forward =====
Input x shape: torch.Size([1, 4, 8])

After linear projection:
Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])

After split_heads:
Q shape: torch.Size([1, 2, 4, 4])
K shape: torch.Size([1, 2, 4, 4])
V shape: torch.Size([1, 2, 4, 4])

After attention:
Output per head shape: torch.Size([1, 2, 4, 4])
Attention weights shape: torch.Size([1, 2, 4, 4])

After combine_heads:
Combined output shape: torch.Size([1, 4, 8])

After final linear projection:
Final output shape: torch.Size([1, 4, 8])
